# Prédire le mot suivant : du modèle bigramme aux Transformers

**Cours Intelligence artificielle · Master 2 · Chapitre 3, apprentissage profond : génération de texte avec les Transformers**

Carnet 08 · durée : environ 1 h 30 · données : 20 phrases inventées pour ce carnet et six citations du domaine public

**Ce que vous allez faire**
- construire, en Python pur et sans Internet, un petit modèle de langage qui prédit le mot suivant à partir de 20 phrases sur la lecture, puis lui faire écrire des phrases ;
- régler la **température** du tirage et voir le texte devenir prévisible, puis surprenant, puis incohérent ;
- essayer de vrais Transformers (en ligne), puis dresser votre liste de vérification pour utiliser un agent conversationnel dans vos études de langue et de littérature.

**Comment utiliser ce carnet**
Exécutez chaque cellule de haut en bas : cliquez sur le bouton ▶ à sa gauche, ou appuyez sur Maj+Entrée. Vous n'avez jamais à écrire de code. Modifiez seulement les champs des cellules intitulées « À MODIFIER ». Dans Colab, le menu **Exécution → Tout exécuter** (*Runtime → Run all*) lance tout le carnet d'un coup.

Avant chaque cellule, notez votre **prédiction** ; après, comparez avec « Ce que vous devez voir ». Les cellules de la partie 3 ont besoin d'Internet (elles fonctionnent dans Colab) ; sans connexion, lisez-les et passez à la partie 4 : rien ensuite n'en dépend.

| Partie | Contenu | Durée |
|---|---|---|
| 1 | Un modèle bigramme fait main | 35 min |
| 2 | La température | 15 min |
| 3 | De vrais Transformers (en ligne) | 15 min |
| 4 | Un agent conversationnel, avec esprit critique | 15 min |
| Bilan | À vous, à rendre | 10 min |

## Partie 1 · Un modèle de langage fait main

Un **modèle de langage** répond à une seule question : *quel mot vient ensuite ?* Le clavier de votre téléphone le fait quand il propose trois mots au-dessus des touches ; GPT le fait aussi, token après token (chapitre 3, « Génération de texte avec les Transformers »).

Nous allons construire le plus simple de tous, le **modèle bigramme**. Un bigramme est une paire de mots voisins : « je lis », « lis un ». Le modèle compte ces paires dans un corpus, puis s'en sert pour prédire et pour écrire.

Le corpus : 20 phrases courtes sur la lecture, les livres, la bibliothèque et la classe, **inventées pour ce carnet** (données fictives), en minuscules et sans ponctuation pour simplifier. Le champ `votre_phrase` permet d'en ajouter une : laissez-le vide pour l'instant.

**Avant d'exécuter :** 20 phrases de 4 à 10 mots… combien de mots au total (les occurrences), selon vous ? Et combien de mots différents ?

In [ ]:
#@title À MODIFIER (facultatif) : le corpus, et votre phrase en plus
corpus = """je lis un roman à la bibliothèque
je lis un poème en classe
je lis un livre le soir
la bibliothèque ouvre le matin
la bibliothèque prête des livres aux étudiants
le professeur lit un poème en classe
le professeur corrige les copies le soir
les étudiants lisent un roman en classe
les étudiants empruntent des livres à la bibliothèque
un roman raconte une histoire
un poème raconte une émotion
la bibliothécaire range les livres le matin
la classe écoute le professeur
le livre est sur la table
le roman est long mais le poème est court
ma sœur lit un roman dans le train
nous lisons le poème à voix haute
le dictionnaire aide les étudiants en classe
chaque livre raconte une histoire
la lecture ouvre l'esprit"""
votre_phrase = "" #@param {type:"string"}

phrases = corpus.split("\n")                    # une phrase par ligne
if votre_phrase != "":
    phrases.append(votre_phrase.lower().strip(" .!?"))
mots = " ".join(phrases).split()                # tous les mots, à la suite
print("Nombre de phrases :", len(phrases))
print("Nombre de mots (occurrences) :", len(mots))
print("Nombre de mots différents :", len(set(mots)))   # set() garde un exemplaire de chaque mot

**Ce que vous devez voir**
```
Nombre de phrases : 20
Nombre de mots (occurrences) : 128
Nombre de mots différents : 53
```
(Un peu plus si vous avez ajouté une phrase.) Comme au chapitre 2, on distingue les **occurrences** (*tokens* : chaque apparition d'un mot) et les **formes** (*types* : les mots différents). Notre modèle ne connaîtra que ces 53 mots.

**Question :** un modèle entraîné sur ce corpus parlera surtout de livres et de classe. Pourquoi ? Que dirait-il s'il avait été entraîné sur des recettes de cuisine ?

### Compter les paires de mots voisins

Pour chaque mot, on note le mot qui le suit, et on compte. On ajoute un point « . » à la fin de chaque phrase : pour le modèle, c'est un mot comme un autre, qui signifie « fin de la phrase ». Ainsi, « je lis un livre le soir » donne six paires : je → lis, lis → un, un → livre, livre → le, le → soir, soir → « . ».

Le résultat est un dictionnaire (chapitre 2) : à chaque mot correspond un `Counter` des mots qui le suivent. Le modèle **n'apprend rien d'autre** que ces effectifs.

**Avant d'exécuter :** relisez le corpus. Quels mots suivent « bibliothèque », et combien de fois chacun ?

In [ ]:
#@title Compter, pour chaque mot, les mots qui le suivent (les bigrammes)
from collections import Counter

def compter_suivants(liste_de_phrases):
    suivants = {}                                    # mot -> Counter des mots suivants
    for phrase in liste_de_phrases:
        mots_phrase = phrase.lower().split() + ["."]   # le point marque la fin
        for i in range(len(mots_phrase) - 1):
            mot, suivant = mots_phrase[i], mots_phrase[i + 1]
            if mot not in suivants:
                suivants[mot] = Counter()
            suivants[mot][suivant] += 1
    return suivants

suivants = compter_suivants(phrases)
print("Mots qui ont au moins un suivant :", len(suivants))
print("Après « bibliothèque » :", suivants["bibliothèque"].most_common())

**Ce que vous devez voir**
```
Mots qui ont au moins un suivant : 53
Après « bibliothèque » : [('.', 2), ('ouvre', 1), ('prête', 1)]
```
« bibliothèque » termine deux phrases (d'où le point, deux fois) et il est suivi une fois de « ouvre », une fois de « prête ». Les 53 mots ont tous un suivant, puisque chaque phrase finit par un point. En tout, le modèle a retenu **92 paires différentes** : ce sont tous ses « paramètres ».

**Question :** ce modèle « sait »-il ce qu'est une bibliothèque ? Que sait-il exactement ?

### Le tableau des mots suivants, et une prédiction

Choisissez un mot dans le menu (ou tapez-en un). La cellule affiche le tableau des mots qui le suivent, avec leur **effectif** et leur **probabilité** : effectif divisé par le total. Le mot suivant le plus probable est celui qui a le plus grand effectif ; en cas d'égalité, Python garde le premier rencontré dans le corpus.

**Avant d'exécuter :** après « un », quel mot vient le plus souvent ? Avec quelle probabilité, à peu près ?

In [ ]:
#@title À MODIFIER : le mot à étudier
mot_choisi = "un" #@param ["un", "le", "la", "les", "je", "lis", "raconte", "est"] {allow-input: true}

compteur = suivants.get(mot_choisi, Counter())    # Counter vide si le mot est inconnu
total = sum(compteur.values())
print(f"« {mot_choisi} » est suivi {total} fois dans le corpus.")
print("   mot suivant    effectif   probabilité")
for suivant, n in compteur.most_common():
    print(f"   {suivant:15} {n:5}      {n / total:.2f}")
if total > 0:
    print("Prédiction : le mot suivant le plus probable est", compteur.most_common(1)[0][0])

**Ce que vous devez voir**
```
« un » est suivi 8 fois dans le corpus.
   mot suivant    effectif   probabilité
   roman               4      0.50
   poème               3      0.38
   livre               1      0.12
Prédiction : le mot suivant le plus probable est roman
```
Python écrit les nombres décimaux avec un point : 0.50 se lit 0,50. Les trois probabilités font 1 (0,125 est arrondi à 0,12).

**Question :** choisissez « la » dans le menu et relancez. Pourquoi « bibliothèque » l'emporte-t-il ? Est-ce une propriété du français ou de *ce* corpus ?

### Écrire une phrase au hasard, avec une graine fixe

Écrire, pour un modèle de langage, c'est boucler : partir d'un mot, **tirer au sort** le suivant *en proportion de ses effectifs* (après « un » : « roman » a 4 chances sur 8, « poème » 3 sur 8, « livre » 1 sur 8), l'ajouter, et recommencer jusqu'au point (au plus 15 mots). Le tirage vient d'un générateur pseudo-aléatoire : avec la même **graine** (*seed*), tout le monde obtient la même phrase ; changez la graine pour en obtenir une autre.

**Avant d'exécuter :** la phrase générée figurera-t-elle telle quelle dans le corpus ? Sera-t-elle correcte en français ? Aura-t-elle un sens ?

In [ ]:
#@title À MODIFIER : générer une phrase au hasard (graine fixe)
mot_depart = "je" #@param ["je", "le", "la", "les", "un", "nous", "ma"]
graine = 9 #@param {type:"integer"}
import random

def generer(modele, mot_depart):
    phrase = [mot_depart]
    while phrase[-1] != "." and len(phrase) < 15:             # jusqu'au point, 15 mots au plus
        compteur = modele[phrase[-1]]
        tirage = random.choices(list(compteur), weights=list(compteur.values()))
        phrase.append(tirage[0])                              # tirage proportionnel aux effectifs
    return phrase

random.seed(graine)                        # même graine, même phrase pour tout le monde
phrase = generer(suivants, mot_depart)
print(" ".join(phrase))
print("Cette phrase est-elle dans le corpus ?", " ".join(phrase[:-1]) in phrases)

**Ce que vous devez voir**
```
je lis un roman dans le professeur .
Cette phrase est-elle dans le corpus ? False
```
Une phrase **nouvelle** (`False` : elle n'est pas dans le corpus), bien construite en apparence… et absurde : on ne lit pas un roman *dans* un professeur.

**Question :** essayez les graines 2, 8 et 36. Lesquelles donnent une phrase correcte et sensée, correcte mais absurde, ou incorrecte ?

### Vérifier la phrase, paire par paire

**Avant d'exécuter :** chaque paire de mots voisins de la phrase générée a-t-elle déjà été vue dans le corpus ? Si oui, combien de fois ?

In [ ]:
#@title Vérifier chaque paire de la phrase générée
for i in range(len(phrase) - 1):
    mot, suivant = phrase[i], phrase[i + 1]
    print(f"{mot:>14} → {suivant:14} vu {suivants[mot][suivant]} fois dans le corpus")

**Ce que vous devez voir**
```
            je → lis            vu 3 fois dans le corpus
           lis → un             vu 3 fois dans le corpus
            un → roman          vu 4 fois dans le corpus
         roman → dans           vu 1 fois dans le corpus
          dans → le             vu 1 fois dans le corpus
            le → professeur     vu 3 fois dans le corpus
    professeur → .              vu 1 fois dans le corpus
```
Chaque paire existe dans le corpus : localement, tout est « juste ». « roman dans le » vient de « ma sœur lit un roman dans le train », « le professeur » de trois autres phrases. C'est l'assemblage qui n'a pas de sens.

**Question :** à quel moment de la phrase le modèle a-t-il « oublié » qu'il parlait d'un roman ?

### Pourquoi ça sonne juste mais ne veut rien dire

- **Un seul mot de contexte.** Le modèle ne regarde que le mot précédent. Après « dans le », il ne sait plus qu'il était question d'un roman.
- **Aucune connaissance du monde.** Il n'a jamais vu de professeur ni de roman, seulement des chaînes de caractères. Il connaît des *cooccurrences*, pas des *significations*. Le linguiste J. R. Firth disait : « You shall know a word by the company it keeps » ; notre modèle ne connaît que la compagnie.
- **Aucune intention.** Il ne veut rien dire : il enchaîne ce qui est plausible d'après ses effectifs.
- **Il invente.** En recombinant des morceaux vus, il produit des phrases qui n'existent nulle part : c'est son apparente créativité, et son danger.

Les grands modèles comme GPT suivent le même principe, prédire le token suivant, mais avec des milliers de tokens de contexte, le mécanisme d'attention et des milliards de paramètres. Leurs phrases sonnent beaucoup plus juste et le sens *semble* présent ; leur objectif reste la plausibilité, pas la vérité. D'où les hallucinations (partie 4).

## Partie 2 · La température

Jusqu'ici, un mot vu 3 fois avait 3 fois plus de chances d'être tiré qu'un mot vu une fois. La **température** $T$ modifie ces chances. Le chapitre 3 donne la formule pour les scores bruts $z_i$ d'un réseau de neurones : $p_i = \exp(z_i / T) \,/\, \sum_j \exp(z_j / T)$. Pour nos effectifs $n_i$ (en prenant $z_i = \ln n_i$), elle revient à élever chaque effectif à la puissance $1/T$, puis à diviser par le total :

$$p_i = \frac{n_i^{1/T}}{\sum_j n_j^{1/T}}$$

- $T = 1$ : les probabilités restent proportionnelles aux effectifs ;
- $T < 1$ : les écarts se creusent, le mot le plus fréquent rafle presque tout (texte prévisible) ;
- $T > 1$ : les écarts s'aplatissent, les mots rares ont leur chance (texte surprenant, puis incohérent).

**Avant d'exécuter :** après « le », « professeur » a 3 chances sur 13, soit 0,23, à $T = 1$. À $T = 0{,}3$, sa probabilité va-t-elle monter ou baisser ? Jusqu'où ?

In [ ]:
#@title À MODIFIER : un mot et une température
mot_etudie = "le" #@param ["le", "un", "la", "les"]
temperature = 0.3 #@param {type:"slider", min:0.1, max:3, step:0.1}

def probabilites(compteur, temperature):
    poids = {}
    for mot, n in compteur.items():
        poids[mot] = n ** (1 / temperature)          # effectif « puissance 1/T »
    total = sum(poids.values())
    return {mot: poids[mot] / total for mot in poids}  # la somme vaut 1

proba_1 = probabilites(suivants[mot_etudie], 1)
proba_t = probabilites(suivants[mot_etudie], temperature)
print(f"Après « {mot_etudie} »    effectif   T = 1   T = {temperature}")
for mot, n in suivants[mot_etudie].most_common():
    print(f"   {mot:15} {n:5}    {proba_1[mot]:.2f}    {proba_t[mot]:.2f}")

**Ce que vous devez voir**
```
Après « le »    effectif   T = 1   T = 0.3
   professeur          3    0.23    0.53
   soir                2    0.15    0.14
   matin               2    0.15    0.14
   poème               2    0.15    0.14
   livre               1    0.08    0.01
   roman               1    0.08    0.01
   train               1    0.08    0.01
   dictionnaire        1    0.08    0.01
```
À $T = 0{,}3$, « professeur » passe de 0,23 à 0,53 : il rafle plus de la moitié des tirages, et les quatre mots vus une seule fois tombent à 0,01.

**Question :** poussez le curseur à 3 et relancez. Que deviennent les écarts entre les mots ? Lequel des deux réglages ressemble le plus à un choix « au hasard complet » ?

### Trois températures côte à côte

Le graphique montre la même distribution des mots suivants à trois températures : 0,3, 1 et 3. Les mots sont rangés du plus fréquent (en haut) au plus rare.

**Avant d'exécuter :** à quoi ressemblera la barre de « professeur » dans le panneau de droite ($T = 3$) ?

In [ ]:
#@title Tracer la distribution à T = 0,3, 1 et 3
import matplotlib.pyplot as plt
mots_suivants = [mot for mot, n in suivants[mot_etudie].most_common()]
fig, axes = plt.subplots(1, 3, figsize=(11, 1.5 + 0.35 * len(mots_suivants)), sharey=True, sharex=True)
for ax, t in zip(axes, [0.3, 1, 3]):
    proba = probabilites(suivants[mot_etudie], t)
    ax.barh(mots_suivants, [proba[mot] for mot in mots_suivants], color="#2a78d6", height=0.7, zorder=3)
    ax.set_title(f"T = {t}")
    ax.set_xlabel("probabilité")
    ax.grid(axis="x", color="#e5e5e5")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].invert_yaxis()                     # le mot le plus fréquent en haut
fig.suptitle(f"Après « {mot_etudie} » : le mot suivant selon la température")
plt.tight_layout()
plt.show()

**Ce que vous devez voir**

Trois panneaux de barres horizontales. À gauche ($T = 0{,}3$), une seule longue barre, « professeur » (0,53), trois barres moyennes (0,14) et quatre barres presque invisibles. Au milieu ($T = 1$), les proportions du corpus. À droite ($T = 3$), des barres presque égales, entre 0,11 et 0,16 : les mots rares sont presque aussi probables que les fréquents.

**Question :** si vous deviez décrire la température à un camarade de lettres sans formule, quelle image ou quelle métaphore choisiriez-vous ?

### Tirer vingt fois le mot suivant

Une probabilité se voit mieux sur des tirages. La cellule tire 20 fois le mot qui suit le mot étudié, à trois températures, avec une graine fixe.

**Avant d'exécuter :** sur 20 tirages à $T = 0{,}2$, combien de fois « professeur », selon vous ? Et à $T = 3$ ?

In [ ]:
#@title Tirer 20 fois le mot suivant, à trois températures
random.seed(1)
for t in [0.2, 1, 3]:
    proba = probabilites(suivants[mot_etudie], t)
    tirages = random.choices(list(proba), weights=list(proba.values()), k=20)
    print(f"T = {t} :", Counter(tirages).most_common())

**Ce que vous devez voir**
```
T = 0.2 : [('professeur', 13), ('soir', 3), ('matin', 2), ('poème', 2)]
T = 1 : [('matin', 7), ('professeur', 5), ('soir', 3), ('livre', 2), ('dictionnaire', 1), ('poème', 1), ('roman', 1)]
T = 3 : [('train', 5), ('professeur', 4), ('poème', 3), ('dictionnaire', 2), ('soir', 2), ('roman', 2), ('livre', 1), ('matin', 1)]
```
À $T = 0{,}2$, « professeur » sort 13 fois sur 20 et seuls quatre mots apparaissent. À $T = 1$, sept mots différents. À $T = 3$, les huit mots sortent, et « train », vu une seule fois dans le corpus, arrive même en tête.

**Question :** pourquoi les effectifs obtenus à $T = 1$ ne sont-ils pas exactement proportionnels à ceux du corpus (3, 2, 2, 2, 1…) ?

### Des phrases entières à la température choisie

Même boucle qu'en partie 1, mais chaque mot est tiré avec les probabilités « à la température $T$ ». On part du mot de départ choisi plus haut (« je »).

**Avant d'exécuter :** à $T = 0{,}2$, les cinq phrases seront-elles différentes les unes des autres ?

In [ ]:
#@title À MODIFIER : cinq phrases à la température choisie
temperature_phrases = 0.2 #@param {type:"slider", min:0.1, max:3, step:0.1}
graine_phrases = 1 #@param {type:"integer"}

random.seed(graine_phrases)
for numero in range(1, 6):
    phrase_t = [mot_depart]
    while phrase_t[-1] != "." and len(phrase_t) < 15:
        proba = probabilites(suivants[phrase_t[-1]], temperature_phrases)
        tirage = random.choices(list(proba), weights=list(proba.values()))
        phrase_t.append(tirage[0])
    print(numero, " ".join(phrase_t))

**Ce que vous devez voir**
```
1 je lis un roman en classe .
2 je lis un roman à la bibliothèque .
3 je lis un roman en classe .
4 je lis un roman dans le professeur corrige les étudiants lisent un roman en classe
5 je lis un roman dans le professeur lit un poème en classe .
```
À $T = 0{,}2$, le modèle prend presque toujours le mot le plus fréquent : « je lis un roman » ouvre les cinq phrases, et deux d'entre elles sont identiques. Le hasard ne joue plus qu'après « roman », dont les cinq suivants sont à égalité (vus une fois chacun) : quand les effectifs sont égaux, la température ne change rien. La phrase 4 atteint 15 mots sans trouver de point.

**Question :** passez le curseur à 3 et relancez. Quelle température choisiriez-vous pour produire des phrases modèles dans un exercice de langue ? Et pour un « cadavre exquis », le jeu d'écriture des surréalistes ?

## Partie 3 · De vrais Transformers (en ligne)

> **Besoin d'une connexion à Internet ; fonctionne dans Colab.** Les cellules de cette partie téléchargent un modèle depuis le Hub de Hugging Face (plusieurs centaines de Mo, une à deux minutes). Sans connexion, lisez-les et passez à la partie 4 : rien ensuite n'en dépend. En local, il faut d'abord `pip install transformers torch` (torch est volumineux).

Un Transformer fait la même chose que notre modèle, prédire un mot à partir des autres, mais à une tout autre échelle :

| | Notre modèle bigramme | CamemBERT (encodeur) | GPT-2 (décodeur, chapitre 3) |
|---|---|---|---|
| Tâche | prédire le mot suivant | deviner un mot caché (texte à trous) | prédire le token suivant |
| Contexte | 1 mot avant | toute la phrase, avant **et** après le trou (512 tokens au plus) | jusqu'à 1 024 tokens avant |
| Unité | le mot | le token (mot ou morceau de mot) | le token |
| Ce qu'il a appris | 92 paires comptées | environ 110 millions de paramètres | 124 millions (82 millions pour distilgpt2) |
| Données | 20 phrases | 138 Go de français tirés du Web (corpus OSCAR) | environ 40 Go de pages Web en anglais |

Les encodeurs comme BERT et CamemBERT lisent la phrase dans les deux sens : on les entraîne à retrouver des mots masqués, exactement comme dans un **test de closure** (texte à trous), bien connu en didactique des langues. Dans une phrase pour CamemBERT, le trou s'écrit `<mask>`.

**Avant d'exécuter :** notre modèle retenait 92 paires. Combien de paramètres la cellule va-t-elle annoncer pour CamemBERT ?

In [ ]:
#@title Télécharger CamemBERT, un modèle « à trous » pour le français (en ligne)
%pip install -q transformers torch
from transformers import pipeline
devineur = pipeline("fill-mask", model="camembert-base")
print("Modèle prêt :", devineur.model.name_or_path)
print("Nombre de paramètres :", devineur.model.num_parameters())

**Ce que vous devez voir**

Après le téléchargement (des barres de progression, parfois des avertissements en rouge : ils sont normaux), deux lignes : le nom du modèle, `camembert-base`, et un nombre de paramètres d'environ 110 millions. Plus d'un million de fois les 92 paires de notre modèle.

**Question :** où et sur quels textes ces paramètres ont-ils été appris ? Quelles variétés de français ce corpus représente-t-il mal, selon vous ?

### Un texte à trous pour CamemBERT

(En ligne : besoin d'Internet ; fonctionne dans Colab.) Choisissez une phrase dans le menu ou tapez la vôtre, avec **un seul** `<mask>`. La cellule affiche les cinq mots les plus probables pour le trou, avec leur score (une probabilité entre 0 et 1).

**Avant d'exécuter :** écrivez d'abord votre propre réponse. Quel mot mettriez-vous à la place de `<mask>` ?

In [ ]:
#@title À MODIFIER : une phrase à trous pour CamemBERT (en ligne)
phrase_a_trou = "Le bibliothécaire range les <mask> sur les étagères." #@param ["Le bibliothécaire range les <mask> sur les étagères.", "Victor Hugo est un <mask> français.", "En classe, les étudiants lisent un <mask> de Maupassant."] {allow-input: true}

for proposition in devineur(phrase_a_trou, top_k=5):
    print(f"{proposition['token_str'].strip():>15}   score {proposition['score']:.2f}")

**Ce que vous devez voir**

Cinq lignes, du mot le plus probable au moins probable, chacune avec un score entre 0 et 1. Pour la première phrase, attendez-vous à des noms pluriels plausibles, très probablement « livres » en tête. Les scores exacts dépendent de la version installée : notez les vôtres.

**Question :** le modèle tient-il compte des mots placés *après* le trou (« sur les étagères ») ? Essayez « Le bibliothécaire range les `<mask>` dans le tiroir. » et comparez avec notre modèle bigramme, qui ne regardait qu'un mot en arrière.

### Et pour générer du français ?

Le modèle génératif du chapitre 3, GPT-2, et sa version allégée distilgpt2 ont été entraînés sur de l'anglais : en français, ils produisent un charabia. De petits modèles génératifs pour le français existent sur le Hub de Hugging Face, mais, à une taille qui tient dans une séance de TP, leurs textes sont pauvres : répétitions, phrases qui se contredisent, français approximatif. Les bons générateurs de français sont des modèles bien plus gros, ceux des agents conversationnels. Pour le français, ce carnet s'en tient donc aux modèles à trous.

Deuxième expérience : le même trou, soumis à un modèle **multilingue**, `bert-base-multilingual-cased`, entraîné sur Wikipédia en 104 langues, dont le français, l'anglais et l'arabe. Attention : pour ce modèle, le trou s'écrit `[MASK]`, et non `<mask>`.

(En ligne : besoin d'Internet ; fonctionne dans Colab. Nouveau téléchargement, plus lourd.)

**Avant d'exécuter :** ce modèle, qui partage ses paramètres entre 104 langues, devinera-t-il mieux ou moins bien que CamemBERT, spécialiste du français ?

In [ ]:
#@title À MODIFIER : le même trou pour un modèle multilingue (en ligne)
phrase_multilingue = "Le bibliothécaire range les [MASK] sur les étagères." #@param ["Le bibliothécaire range les [MASK] sur les étagères.", "Victor Hugo est un [MASK] français.", "The librarian puts the [MASK] on the shelves."] {allow-input: true}

from transformers import pipeline
devineur_multilingue = pipeline("fill-mask", model="bert-base-multilingual-cased")
for proposition in devineur_multilingue(phrase_multilingue, top_k=5):
    print(f"{proposition['token_str'].strip():>15}   score {proposition['score']:.2f}")

**Ce que vous devez voir**

Cinq propositions avec leur score, comme pour CamemBERT. Le modèle multilingue découpe les mots français en morceaux plus petits : il propose parfois des fragments de mots (précédés de `##`), et ses scores sont souvent plus dispersés. Les valeurs exactes dépendent de la version installée.

**Question :** lequel des deux modèles devine le mieux ? Essayez la phrase anglaise du menu, puis collez une phrase en arabe, par exemple : يقرأ الطالب [MASK] في المكتبة. Que peut-on en conclure sur les langues peu présentes sur le Web ?

## Partie 4 · Utiliser un agent conversationnel, avec esprit critique

Un agent conversationnel (ChatGPT et ses semblables) est un très grand modèle de langage, ajusté pour dialoguer : il prolonge votre texte par la suite la plus plausible. **Plausible ne veut pas dire vrai.** Voici la liste de vérification à garder sous les yeux pendant vos études de langue et de littérature.

1. **Références inventées.** Un agent peut fournir une référence complète (auteur réel, titre vraisemblable, éditeur connu, année) qui n'existe pas. → Ne citez qu'un ouvrage retrouvé dans un catalogue de bibliothèque ou chez l'éditeur, et lu par vous.
2. **Citations inventées ou déformées.** Une « citation » de Hugo ou de Shakespeare peut être fausse, retouchée ou attribuée au mauvais auteur. → Vérifiez chaque citation dans l'édition d'origine, page comprise (pour les textes anciens : Gallica, Wikisource, Project Gutenberg).
3. **Explications de langue.** Règle de grammaire, étymologie, sens d'un mot : l'agent répond avec assurance, parfois à tort. → Confrontez à une grammaire de référence ou à un dictionnaire.
4. **Corrections et traductions.** L'agent peut « corriger » ce qui était juste, effacer votre style ou imposer une seule variété de la langue. → Demandez l'explication de chaque correction ; ne gardez que celles que vous comprenez.
5. **Données et textes des autres.** Ne collez ni réponses d'enquête, ni copies d'apprenants, ni textes inédits ou protégés.
6. **Usage déclaré.** Déclarez tout usage à la fin du travail rendu : outil, date, usage, demandes recopiées en annexe, vérification. Rendre comme sien un texte généré relève du plagiat.

Ces règles sont celles de la charte d'usage du cours : [L'IA générative dans votre classe](https://progremer04.github.io/canva-au-for-ang-/enseignant.html#guide-ia) (guide, section « Une charte d'usage pour la classe »). Voir aussi les leçons [Perspectives, limites et éthique](https://progremer04.github.io/canva-au-for-ang-/#c1-perspectives) et [Apprendre avec un assistant d'IA, sans tricher](https://progremer04.github.io/canva-au-for-ang-/#me-assistant).

### D'où vient une fausse citation ?

Reprenons notre modèle bigramme, entraîné cette fois sur six vraies citations du domaine public : Hugo (« Demain, dès l'aube… », *Les Contemplations*, 1856), Verlaine (*Romances sans paroles*, 1874), Baudelaire (« Correspondances », *Les Fleurs du mal*, 1857), Pascal (*Pensées*, 1670), La Fontaine (« Le Loup et l'Agneau », 1668) et Rimbaud (lettre à Paul Demeny, 1871). On part du premier mot de la citation d'un auteur, on laisse le modèle continuer, puis on attribue le résultat à cet auteur, comme le ferait une réponse trop sûre d'elle.

**Avant d'exécuter :** la « citation » obtenue existera-t-elle ? Sonnera-t-elle comme du Pascal ?

In [ ]:
#@title À MODIFIER : fabriquer une fausse citation
citations = {"Victor Hugo": "demain dès l'aube à l'heure où blanchit la campagne je partirai",
             "Paul Verlaine": "il pleure dans mon cœur comme il pleut sur la ville",
             "Charles Baudelaire": "la nature est un temple où de vivants piliers laissent parfois sortir de confuses paroles",
             "Blaise Pascal": "le cœur a ses raisons que la raison ne connaît point",
             "Jean de La Fontaine": "la raison du plus fort est toujours la meilleure",
             "Arthur Rimbaud": "je est un autre"}   # six vraies citations, domaine public
auteur = "Blaise Pascal" #@param ["Victor Hugo", "Paul Verlaine", "Charles Baudelaire", "Blaise Pascal", "Jean de La Fontaine", "Arthur Rimbaud"]
graine_citation = 1 #@param {type:"integer"}

modele_citations = compter_suivants(list(citations.values()))
random.seed(graine_citation)
fausse = generer(modele_citations, citations[auteur].split()[0])
texte = " ".join(fausse[:-1])
print(f"« {texte.capitalize()} » ({auteur})")
print("Cette citation existe-t-elle vraiment ?", texte in citations.values())

**Ce que vous devez voir**
```
« Le cœur a ses raisons que la raison du plus fort est un autre » (Blaise Pascal)
Cette citation existe-t-elle vraiment ? False
```
Une phrase fluide, au rythme de maxime, signée d'un vrai nom… que Pascal n'a jamais écrite : le début vient de Pascal, le milieu de La Fontaine (« la raison du plus fort est »), la fin de Rimbaud (« un autre »). Chaque paire de mots est authentique ; l'ensemble est faux. Un grand modèle fait la même chose, de façon beaucoup plus convaincante : c'est ainsi que naissent les citations et les références inventées.

**Question :** sans la dernière ligne, comment auriez-vous pu démasquer cette fausse citation ? Où iriez-vous la vérifier ?

### Votre liste de vérification et votre déclaration d'usage

Pensez à la dernière fois que vous avez utilisé un agent conversationnel pour vos études. Remplissez les champs (outil, usage, vérification) et cochez seulement ce que vous avez **vraiment** fait. La cellule affiche votre liste et une déclaration d'usage sur le modèle du guide, à recopier à la fin de votre travail.

**Avant d'exécuter :** combien de cases pourrez-vous cocher honnêtement ?

In [ ]:
#@title À MODIFIER : ma liste de vérification et ma déclaration d'usage
outil = "un agent conversationnel (version gratuite en ligne, octobre 2026)" #@param {type:"string"}
usage = "me faire expliquer la notion de température" #@param {type:"string"}
verification = "j'ai comparé la réponse avec la leçon du chapitre 3" #@param {type:"string"}
references_retrouvees = False #@param {type:"boolean"}
citations_verifiees = False #@param {type:"boolean"}
demandes_en_annexe = False #@param {type:"boolean"}

controles = {"Références : chacune retrouvée dans un catalogue (ou aucune)": references_retrouvees,
             "Citations : chacune vérifiée dans l'édition d'origine (ou aucune)": citations_verifiees,
             "Mes demandes recopiées en annexe": demandes_en_annexe}
for intitule, fait in controles.items():
    print("[x]" if fait else "[ ] À FAIRE :", intitule)
print(f"\n« J'ai utilisé {outil} pour {usage}. Mes demandes figurent en annexe. "
      f"Vérification : {verification}. L'analyse et la rédaction sont de moi. »")

**Ce que vous devez voir** (avec les valeurs par défaut)
```
[ ] À FAIRE : Références : chacune retrouvée dans un catalogue (ou aucune)
[ ] À FAIRE : Citations : chacune vérifiée dans l'édition d'origine (ou aucune)
[ ] À FAIRE : Mes demandes recopiées en annexe

« J'ai utilisé un agent conversationnel (version gratuite en ligne, octobre 2026) pour me faire expliquer la notion de température. Mes demandes figurent en annexe. Vérification : j'ai comparé la réponse avec la leçon du chapitre 3. L'analyse et la rédaction sont de moi. »
```
Tant qu'une ligne affiche « À FAIRE », la déclaration n'est pas encore vraie : faites la vérification, ou retirez l'élément de votre travail.

**Question :** laquelle des six règles de la liste vous paraît la plus difficile à respecter dans vos études, et pourquoi ?

## À vous

Trois modifications à faire avec les champs, sans écrire de code :

1. **Enrichir le corpus.** Dans la première cellule, tapez une phrase dans `votre_phrase` (par exemple « le professeur lit un roman le matin »), puis **Exécution → Tout exécuter**. Le tableau de « le » a-t-il changé ? La phrase de la graine 9 aussi ? Pourquoi ?
2. **Inventer ou recopier ?** Dans « générer une phrase au hasard », essayez dix graines (1 à 10). Combien de phrases sont nouvelles (`False`) ? Combien ont un sens ?
3. **Température.** Dans la partie 2, faites glisser `temperature_phrases` de 0,1 à 3, par pas de 0,5, et notez à chaque fois la phrase la plus étrange. En ligne, proposez à CamemBERT trois phrases à trous tirées d'un de vos cours de langue et comparez ses réponses aux vôtres.

## Pour le mini-projet

Le modèle bigramme est un outil d'exploration : les paires les plus fréquentes d'un corpus en révèlent les tics et les formules. Essayez-le sur vos propres données : avis de lecteurs (sujet 1), réponses ouvertes d'un questionnaire (sujet 2), discours politiques (sujet 3). Préparez un fichier **CSV en UTF-8** avec une colonne intitulée `texte`, une phrase par ligne (un tableur fait cela très bien). Dans Colab, cochez la case, exécutez la cellule et choisissez votre fichier ; dans Jupyter, placez le fichier `mon_corpus.csv` à côté du carnet.

**Avant d'exécuter :** tant que la case n'est pas cochée, que va faire la cellule ?

In [ ]:
#@title Pour le mini-projet : votre propre corpus (facultatif)
charger_mon_fichier = False #@param {type:"boolean"}
mot_a_etudier = "la" #@param {type:"string"}
import pandas as pd
if charger_mon_fichier:
    try:
        from google.colab import files
        nom_fichier = list(files.upload())[0]   # Colab : choisissez votre fichier CSV
    except ImportError:
        nom_fichier = "mon_corpus.csv"          # Jupyter : fichier placé à côté du carnet
    tableau = pd.read_csv(nom_fichier)          # une colonne « texte », une phrase par ligne
    mon_modele = compter_suivants(list(tableau["texte"].astype(str)))
    print(f"Après « {mot_a_etudier} » :", mon_modele.get(mot_a_etudier, Counter()).most_common(10))
else:
    print("Aucun fichier chargé : cochez la case, puis relancez la cellule.")

**Ce que vous devez voir**
```
Aucun fichier chargé : cochez la case, puis relancez la cellule.
```
Avec votre fichier : les dix mots qui suivent le plus souvent le mot étudié dans votre corpus, avec leurs effectifs. Pour un corpus réel, supprimez d'abord la ponctuation, ou acceptez que « ville, » et « ville » comptent comme deux mots.

**Question :** dans votre corpus, quel mot aimeriez-vous étudier ainsi, et qu'espérez-vous y découvrir ?

## À rendre

Répondez par écrit, en français, en 8 à 12 lignes pour chaque question.

1. Recopiez une phrase produite par le modèle bigramme (partie 1) qui « sonne juste » mais ne veut rien dire. À l'aide de la vérification paire par paire, expliquez pourquoi le modèle l'a produite et ce qui lui manque pour avoir du sens.
2. Comparez les phrases obtenues à $T = 0{,}2$ et à $T = 3$ (partie 2). Quelle température choisiriez-vous pour un exercice de langue, et laquelle pour un atelier d'écriture ? Justifiez avec vos propres résultats.
3. À partir de la fausse citation de la partie 4, expliquez à un camarade pourquoi un agent conversationnel peut inventer une citation ou une référence. Terminez par vos trois vérifications personnelles et par votre déclaration d'usage pour le mini-projet.

## Ce qu'il faut retenir

- Un modèle de langage prédit le mot suivant à partir de ce qu'il a vu. Notre modèle bigramme compte des paires de mots ; les Transformers font de même avec des tokens, l'attention, un long contexte et des millions de paramètres.
- La température règle le tirage : basse, le texte est prévisible ; haute, il devient surprenant, puis incohérent. Chaque paire peut être juste et la phrase absurde : plausible ne veut pas dire vrai.
- Avec un agent conversationnel : retrouvez chaque référence, vérifiez chaque citation dans l'édition d'origine, déclarez tout usage.

Leçon du cours : [Chapitre 3 · L'apprentissage profond et les Transformers](https://progremer04.github.io/canva-au-for-ang-/#c3-deep) · Charte d'usage : [L'IA générative dans votre classe](https://progremer04.github.io/canva-au-for-ang-/enseignant.html#guide-ia)